# Deforestation Detector — MLP Classifier

Loads pre-built training arrays from `build_dataset.py`, trains a temporal MLP
on 6-year AEF embeddings (384 dims), and generates `submission/submission.geojson`.

**Run order:**
1. `python fuse_labels.py`
2. `train_model1.ipynb`  (optional but recommended)
3. `python build_dataset.py --labels combined`
4. Run this notebook top to bottom


In [2]:
# ── Cell 2: Imports ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import datetime
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.features import geometry_mask
from pathlib import Path
from tqdm.auto import tqdm
import geopandas as gpd

c:\Users\khess\anaconda3\envs\deep_learning\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ── Cell 3: Constants & dequantization (straight from workshop) ────────────────
POWER  = 2.0
SCALE  = 127.5
OFFSET = 127

def set_nan_on_zero_groups(emb, group_size=64, inplace=True):
    out = emb if inplace else emb.copy()
    grouped = out.reshape(out.shape[0] // group_size, group_size, -1)
    zero_groups = np.all(grouped == 0, axis=1)
    grouped[:] = np.where(zero_groups[:, None, :], np.nan, grouped)
    return out

def dequantize_embeddings(emb):
    out = emb.astype(np.float32)
    set_nan_on_zero_groups(out, group_size=64, inplace=True)
    out = (out - OFFSET) / SCALE
    neg = out < 0
    out = np.abs(out) ** POWER
    out[neg] *= -1
    return out

In [4]:
# ── Cell 4: Data paths & discover all tiles ────────────────────────────────────
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from config import AEF_TRAIN, AEF_TEST, RADD_DIR, GLADS2_DIR

YEARS = ['2020', '2021', '2022', '2023', '2024', '2025']

_radd_dir   = Path(RADD_DIR)
_glads2_dir = Path(GLADS2_DIR)

def get_tile_paths(tile_id, split='train'):
    paths = {}
    if split == 'train':
        paths['radd']        = _radd_dir   / f'radd_{tile_id}_labels.tif'
        paths['glads2']      = _glads2_dir / f'glads2_{tile_id}_alert.tif'
        paths['glads2_date'] = _glads2_dir / f'glads2_{tile_id}_alertDate.tif'
    aef_dir = AEF_TRAIN if split == 'train' else AEF_TEST
    for year in YEARS:
        paths[f'aef_{year}'] = Path(aef_dir) / f'{tile_id}_{year}.tiff'
    return paths

# Discover tiles from AEF train directory
aef_train_dir = Path(AEF_TRAIN)
ALL_TILE_IDS  = sorted(set(
    p.stem.rsplit('_', 1)[0] for p in aef_train_dir.glob('*.tiff')
))

# Split into full-consensus vs RADD-only (GLAD-S2 absent for some tiles)
valid_tiles     = []  # RADD + GLAD-S2 + 6 AEF years
radd_only_tiles = []  # RADD + 6 AEF years (no GLAD-S2)

for tid in ALL_TILE_IDS:
    paths   = get_tile_paths(tid)
    aef_ok  = all(paths[f'aef_{y}'].exists() for y in YEARS)
    radd_ok = paths['radd'].exists()
    g_ok    = paths['glads2'].exists() and paths['glads2_date'].exists()
    if aef_ok and radd_ok and g_ok:
        valid_tiles.append(tid)
    elif aef_ok and radd_ok:
        radd_only_tiles.append(tid)

all_train_tiles = valid_tiles + radd_only_tiles

print(f'Full consensus tiles (RADD + GLAD-S2): {len(valid_tiles)}')
print(f'RADD-only tiles                       : {len(radd_only_tiles)}')
print(f'Total training tiles                  : {len(all_train_tiles)}')
print(f'Feature dims per pixel: {len(YEARS)} years × 64 = {len(YEARS) * 64}')

TILE_ID = all_train_tiles[0]


Full consensus tiles (RADD + GLAD-S2): 8
RADD-only tiles                       : 8
Total training tiles                  : 16
Feature dims per pixel: 6 years × 64 = 384


In [5]:
# ── Cell 5: Load full temporal AEF stack for one tile ─────────────────────────
def load_aef_stack(tile_id, split='train'):
    """Returns (6, 64, H, W) float32 — one 64-dim embedding per year per pixel."""
    paths = get_tile_paths(tile_id, split)
    stack = []
    transform, crs, shape = None, None, None
    for year in YEARS:
        with rasterio.open(paths[f'aef_{year}']) as src:
            raw = src.read()                        # (64, H, W) uint8
            if transform is None:
                transform = src.transform
                crs       = src.crs
                shape     = src.shape
        stack.append(dequantize_embeddings(raw))    # (64, H, W) float32
    return np.stack(stack, axis=0), transform, crs, shape  # (6, 64, H, W)

In [6]:
# ── Cell 6: Load pre-built training dataset ────────────────────────────────────
import numpy as np
from pathlib import Path

LABEL_SOURCE = 'combined'  # 'combined' | 'fused' | 'model1'

x_path = Path(f'outputs/X_train_{LABEL_SOURCE}.npy')
y_path = Path(f'outputs/y_train_{LABEL_SOURCE}.npy')

if not x_path.exists():
    raise FileNotFoundError(
        f'{x_path} not found.\n'
        f'Run: python build_dataset.py --labels {LABEL_SOURCE}'
    )

X     = np.load(x_path)
y_bin = np.load(y_path).astype(np.int32)

n_pos = int(y_bin.sum())
print(f'Loaded  [{LABEL_SOURCE}]  X={X.shape}  positives={n_pos:,} ({100*y_bin.mean():.1f}%)')


Loaded  [combined]  X=(2682676, 384)  positives=670,669 (25.0%)


In [7]:
# ── Train temporal MLP ─────────────────────────────────────────────────────────
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X_train, X_val, y_train, y_val = train_test_split(
    X, y_bin, test_size=0.2, random_state=42, stratify=y_bin)

model = Pipeline([
    ('scaler', StandardScaler()),
    ('mlp',    MLPClassifier(
                   hidden_layer_sizes=(256, 128, 64),
                   activation='relu',
                   batch_size=4096,
                   max_iter=50,
                   early_stopping=True,
                   validation_fraction=0.1,
                   random_state=42,
                   verbose=True))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_val)
print(classification_report(y_val, y_pred,
      target_names=['no deforestation', 'deforestation']))


Iteration 1, loss = 0.13541915
Validation score: 0.952100
Iteration 2, loss = 0.10670883
Validation score: 0.955180
Iteration 3, loss = 0.10025828
Validation score: 0.958600
Iteration 4, loss = 0.09607872
Validation score: 0.959965
Iteration 5, loss = 0.09306863
Validation score: 0.960776
Iteration 6, loss = 0.09060594
Validation score: 0.961107
Iteration 7, loss = 0.08841376
Validation score: 0.960548
Iteration 8, loss = 0.08676909
Validation score: 0.961205
Iteration 9, loss = 0.08496289
Validation score: 0.962481
Iteration 10, loss = 0.08340810
Validation score: 0.962481
Iteration 11, loss = 0.08211920
Validation score: 0.963777
Iteration 12, loss = 0.08070066
Validation score: 0.963945
Iteration 13, loss = 0.07966584
Validation score: 0.964168
Iteration 14, loss = 0.07845749
Validation score: 0.965058
Iteration 15, loss = 0.07758752
Validation score: 0.964066
Iteration 16, loss = 0.07658705
Validation score: 0.964108
Iteration 17, loss = 0.07545376
Validation score: 0.964164
Iterat

In [ ]:
# ── Cell 10: Save trained model ────────────────────────────────────────────────
import sys
import pickle

model_out = Path('outputs/model2_sklearn.pkl')
model_out.parent.mkdir(exist_ok=True)
with open(model_out, 'wb') as f:
    pickle.dump(model, f)
print(f'Model saved -> {model_out}')


Model saved -> outputs\model2_sklearn.pkl


In [21]:
# ── Cell 12: Generate FINAL submission GeoJSON (self-contained, runs standalone)
import sys, json, tempfile, pickle
import numpy as np
import rasterio
from pathlib import Path
from tqdm.auto import tqdm

sys.path.insert(0, str(Path('.').resolve()))
from submission_utils import raster_to_geojson
from config import AEF_TEST

# Load saved model (works even if earlier cells haven't run)
with open('outputs/model2_sklearn.pkl', 'rb') as f:
    model = pickle.load(f)
print('Model loaded.')

YEARS          = ['2020', '2021', '2022', '2023', '2024', '2025']
SUBMISSION_DIR = Path('submission')
SUBMISSION_DIR.mkdir(exist_ok=True)


def _dequantize(raw):
    out = raw.astype(np.float32)
    out = np.where(np.all(out == 0, axis=0, keepdims=True), np.nan, out)
    out = (out - 127.0) / 127.5
    neg = out < 0
    out = np.abs(out) ** 2.0
    out[neg] *= -1.0
    return out


def load_test_aef_stack(tile_id):
    aef_dir = Path(AEF_TEST)
    stack, transform, crs, shape = [], None, None, None
    for year in YEARS:
        p = aef_dir / f'{tile_id}_{year}.tiff'
        if not p.exists():
            return None, None, None, None
        with rasterio.open(p) as src:
            raw = src.read()
            if transform is None:
                transform, crs, shape = src.transform, src.crs, src.shape
        stack.append(_dequantize(raw))
    return np.stack(stack, axis=0), transform, crs, shape  # (6, 64, H, W)


def estimate_month_map(aef_stack):
    diffs = np.stack([
        np.sqrt(np.sum((aef_stack[y + 1] - aef_stack[y]) ** 2, axis=0))
        for y in range(len(YEARS) - 1)
    ])
    change_year = 2021 + np.argmax(diffs, axis=0)
    return ((change_year - 2020) * 12 + 5).astype(np.int16)


def enrich_time_steps(geojson_dict, month_map, transform, native_crs):
    import geopandas as gpd
    from shapely.geometry import shape as shp
    from rasterio.features import geometry_mask
    features = geojson_dict.get('features', [])
    if not features:
        return []
    gdf = gpd.GeoDataFrame(
        geometry=[shp(f['geometry']) for f in features], crs='EPSG:4326'
    ).to_crs(native_crs)
    valid = []
    for i, feat in enumerate(features):
        if feat['geometry']['type'] not in ('Polygon', 'MultiPolygon'):
            continue
        try:
            mask = geometry_mask(
                [gdf.geometry.iloc[i].__geo_interface__],
                transform=transform, invert=True, out_shape=month_map.shape,
            )
            vm = month_map[mask]
            vm = vm[vm >= 0]
            if len(vm):
                pm = int(np.median(vm))
                yr, mo = 2020 + pm // 12, 1 + pm % 12
                feat['properties'] = {'time_step': f'{yr % 100:02d}{mo:02d}'}  # YYMM
            else:
                feat['properties'] = {'time_step': None}
        except Exception:
            feat['properties'] = {'time_step': None}
        valid.append(feat)
    return valid


# ── Inference on test tiles ────────────────────────────────────────────────────
test_aef_dir = Path(AEF_TEST)
TEST_TILES   = sorted(set(p.stem.rsplit('_', 1)[0] for p in test_aef_dir.glob('*.tif*')))
print(f'Test tiles found: {len(TEST_TILES)}')

all_features = []
for tile_id in tqdm(TEST_TILES, desc='Inference'):
    stack, t, crs, shape = load_test_aef_stack(tile_id)
    if stack is None:
        print(f'{tile_id}: missing AEF, skipped')
        continue

    flat      = stack.reshape(len(YEARS) * 64, -1).T
    valid_idx = np.isfinite(flat).all(axis=1)
    proba     = np.zeros(flat.shape[0], dtype=np.float32)
    proba[valid_idx] = model.predict_proba(flat[valid_idx])[:, 1]
    pred_map  = (proba.reshape(shape) > 0.5).astype(np.uint8)

    with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmp:
        tmp_path = Path(tmp.name)
    with rasterio.open(tmp_path, 'w', driver='GTiff',
                       height=shape[0], width=shape[1],
                       count=1, dtype='uint8', crs=crs, transform=t) as dst:
        dst.write(pred_map, 1)

    try:
        geojson   = raster_to_geojson(tmp_path, output_path=None, min_area_ha=0.5)
        month_map = estimate_month_map(stack)
        feats     = enrich_time_steps(geojson, month_map, t, crs)
        all_features.extend(feats)
        print(f'{tile_id}: {len(feats):,} polygons')
    except ValueError:
        print(f'{tile_id}: 0 polygons')
    finally:
        tmp_path.unlink(missing_ok=True)

out_path = SUBMISSION_DIR / 'submission.geojson'
with open(out_path, 'w') as f:
    json.dump({'type': 'FeatureCollection', 'features': all_features}, f)
print(f'\nSUBMISSION READY: {out_path}  ({len(all_features)} total polygons)')


Model loaded.
Test tiles found: 5


Inference:   0%|          | 0/5 [00:00<?, ?it/s]c:\Users\khess\anaconda3\envs\deep_learning\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Inference:  20%|██        | 1/5 [00:18<01:13, 18.29s/it]

18NVJ_1_6: 12 polygons


c:\Users\khess\anaconda3\envs\deep_learning\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Inference:  40%|████      | 2/5 [00:38<00:59, 19.69s/it]

18NYH_2_1: 225 polygons


...

c:\Users\khess\anaconda3\envs\deep_learning\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Inference:  80%|████████  | 4/5 [01:15<00:18, 18.90s/it]

47QMA_6_2: 411 polygons


c:\Users\khess\anaconda3\envs\deep_learning\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
Inference: 100%|██████████| 5/5 [01:36<00:00, 19.22s/it]

48PWA_0_6: 418 polygons



SUBMISSION READY: submission\submission.geojson  (1217 total polygons)
